In [1]:
!pip install -q -U earthengine-api geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.5/481.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 44.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.
google-genai 2.12.1 requires google-auth[requests]<2.56.0,>=2.48.1, but you have google-auth 2.56.3 which is incompatible.


In [2]:
import ee
import geemap
import matplotlib.pyplot as plt

In [3]:
ee.Authenticate()

ee.Initialize(project='fourth-groove-470214-u2')

print("Earth Engine initialized successfully!")

Earth Engine initialized successfully!


In [4]:
study_area = ee.Geometry.Rectangle([
    78.20, 17.25,
    78.65, 17.60
])

print("Study area created!")

Study area created!


In [5]:
def mask_s2_clouds(image):
    """
    Masks clouds and cirrus in Sentinel-2 imagery
    using the QA60 band.
    """

    qa = image.select('QA60')

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(
            qa.bitwiseAnd(cirrus_bit_mask).eq(0)
        )
    )

    return image.updateMask(mask).divide(10000)

In [6]:
def get_sentinel_composite(start_date, end_date, region):

    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(mask_s2_clouds)
    )

    print(
        start_date,
        "to",
        end_date,
        "| Images:",
        collection.size().getInfo()
    )

    composite = collection.median().clip(region)

    return composite

In [7]:
image_2020 = get_sentinel_composite(
    '2020-01-01',
    '2020-01-31',
    study_area
)

2020-01-01 to 2020-01-31 | Images: 8


In [8]:
image_2025 = get_sentinel_composite(
    '2025-01-01',
    '2025-01-31',
    study_area
)

2025-01-01 to 2025-01-31 | Images: 7


In [9]:
print("2020 bands:")
print(image_2020.bandNames().getInfo())

print("\n2025 bands:")
print(image_2025.bandNames().getInfo())

2020 bands:
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']

2025 bands:
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


###NDVI calculating

In [10]:
ndvi_2020 = image_2020.normalizedDifference(
    ['B8', 'B4']
).rename('NDVI')

ndvi_2025 = image_2025.normalizedDifference(
    ['B8', 'B4']
).rename('NDVI')

print("NDVI calculated for 2020 and 2025.")

NDVI calculated for 2020 and 2025.


##NDWI calculatin

In [11]:
ndwi_2020 = image_2020.normalizedDifference(
    ['B3', 'B8']
).rename('NDWI')

ndwi_2025 = image_2025.normalizedDifference(
    ['B3', 'B8']
).rename('NDWI')

print("NDWI calculated for 2020 and 2025.")

NDWI calculated for 2020 and 2025.


##NDBI calculatin

In [12]:
ndbi_2020 = image_2020.normalizedDifference(
    ['B11', 'B8']
).rename('NDBI')

ndbi_2025 = image_2025.normalizedDifference(
    ['B11', 'B8']
).rename('NDBI')

print("NDBI calculated for 2020 and 2025.")

NDBI calculated for 2020 and 2025.


##Add indices to the images

In [13]:
features_2020 = image_2020.addBands([
    ndvi_2020,
    ndwi_2020,
    ndbi_2020
])

features_2025 = image_2025.addBands([
    ndvi_2025,
    ndwi_2025,
    ndbi_2025
])

print("2020 feature bands:")
print(features_2020.bandNames().getInfo())

print("\n2025 feature bands:")
print(features_2025.bandNames().getInfo())

2020 feature bands:
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE', 'NDVI', 'NDWI', 'NDBI']

2025 feature bands:
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE', 'NDVI', 'NDWI', 'NDBI']


##Calculate temporal differences

In [14]:
delta_ndvi = ndvi_2025.subtract(ndvi_2020).rename('Delta_NDVI')

delta_ndwi = ndwi_2025.subtract(ndwi_2020).rename('Delta_NDWI')

delta_ndbi = ndbi_2025.subtract(ndbi_2020).rename('Delta_NDBI')

print("Temporal differences calculated.")

Temporal differences calculated.


In [15]:
temporal_features = (
    features_2020
    .addBands(features_2025, overwrite=False)
    .addBands([
        delta_ndvi,
        delta_ndwi,
        delta_ndbi
    ])
)

print("Complete feature stack created.")

print(
    "\nFeature bands:"
)

print(
    temporal_features.bandNames().getInfo()
)

Complete feature stack created.

Feature bands:
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE', 'NDVI', 'NDWI', 'NDBI', 'B1_1', 'B2_1', 'B3_1', 'B4_1', 'B5_1', 'B6_1', 'B7_1', 'B8_1', 'B8A_1', 'B9_1', 'B11_1', 'B12_1', 'AOT_1', 'WVP_1', 'SCL_1', 'TCI_R_1', 'TCI_G_1', 'TCI_B_1', 'MSK_CLDPRB_1', 'MSK_SNWPRB_1', 'QA10_1', 'QA20_1', 'QA60_1', 'MSK_CLASSI_OPAQUE_1', 'MSK_CLASSI_CIRRUS_1', 'MSK_CLASSI_SNOW_ICE_1', 'NDVI_1', 'NDWI_1', 'NDBI_1', 'Delta_NDVI', 'Delta_NDWI', 'Delta_NDBI']


##vizualizing 2020 ndvi score

In [16]:
ndvi_vis = {
    'min': -1,
    'max': 1,
    'palette': [
        'brown',
        'yellow',
        'green'
    ]
}

m = geemap.Map()

m.centerObject(study_area, 10)

m.addLayer(
    ndvi_2020,
    ndvi_vis,
    'NDVI 2020'
)

m.addLayer(
    study_area,
    {},
    'Study Area'
)

m

Map(center=[17.425070303037895, 78.42499999999899], controls=(WidgetControl(options=['position', 'transparent_…

## 2025 ndvi

In [17]:
m = geemap.Map()

m.centerObject(study_area, 10)

m.addLayer(
    ndvi_2025,
    ndvi_vis,
    'NDVI 2025'
)

m.addLayer(
    study_area,
    {},
    'Study Area'
)

m

Map(center=[17.425070303037895, 78.42499999999899], controls=(WidgetControl(options=['position', 'transparent_…

##2020 and 2025 ndwi

In [18]:
ndwi_vis = {
    'min': -1,
    'max': 1,
    'palette': [
        'brown',
        'white',
        'blue'
    ]
}

m = geemap.Map()

m.centerObject(study_area, 10)

m.addLayer(
    ndwi_2020,
    ndwi_vis,
    'NDWI 2020'
)

m.addLayer(
    ndwi_2025,
    ndwi_vis,
    'NDWI 2025'
)

m.addLayer(
    study_area,
    {},
    'Study Area'
)

m

Map(center=[17.425070303037895, 78.42499999999899], controls=(WidgetControl(options=['position', 'transparent_…

## 2020 and 2025 ndbi

In [19]:
ndbi_vis = {
    'min': -1,
    'max': 1,
    'palette': [
        'blue',
        'white',
        'red'
    ]
}

m = geemap.Map()

m.centerObject(study_area, 10)

m.addLayer(
    ndbi_2020,
    ndbi_vis,
    'NDBI 2020'
)

m.addLayer(
    ndbi_2025,
    ndbi_vis,
    'NDBI 2025'
)

m.addLayer(
    study_area,
    {},
    'Study Area'
)

m

Map(center=[17.425070303037895, 78.42499999999899], controls=(WidgetControl(options=['position', 'transparent_…

##Visualize the actual changes

In [20]:
change_vis = {
    'min': -0.5,
    'max': 0.5,
    'palette': [
        'blue',
        'white',
        'red'
    ]
}

m = geemap.Map()

m.centerObject(study_area, 10)

m.addLayer(
    delta_ndvi,
    change_vis,
    'ΔNDVI: 2025 - 2020'
)

m.addLayer(
    delta_ndwi,
    change_vis,
    'ΔNDWI: 2025 - 2020'
)

m.addLayer(
    delta_ndbi,
    change_vis,
    'ΔNDBI: 2025 - 2020'
)

m.addLayer(
    study_area,
    {},
    'Study Area'
)

m

Map(center=[17.425070303037895, 78.42499999999899], controls=(WidgetControl(options=['position', 'transparent_…

## feature statistics

In [21]:
stats = temporal_features.select([
    'NDVI',
    'NDWI',
    'NDBI'
]).reduceRegion(
    reducer=ee.Reducer.mean()
        .combine(
            reducer2=ee.Reducer.minMax(),
            sharedInputs=True
        ),
    geometry=study_area,
    scale=100,
    maxPixels=1e9
)

print(stats.getInfo())

{'NDBI_max': 0.34251359702069256, 'NDBI_mean': 0.06341963983354708, 'NDBI_min': -0.7689769048776107, 'NDVI_max': 0.8367277844952777, 'NDVI_mean': 0.2830944745107739, 'NDVI_min': -0.3078281415526961, 'NDWI_max': 0.4744661191343304, 'NDWI_mean': -0.3381470426850073, 'NDWI_min': -0.7549063175326726}


##temporal diff statistics

In [22]:
change_stats = ee.Image.cat([
    delta_ndvi,
    delta_ndwi,
    delta_ndbi
]).reduceRegion(
    reducer=ee.Reducer.mean()
        .combine(
            reducer2=ee.Reducer.minMax(),
            sharedInputs=True
        ),
    geometry=study_area,
    scale=100,
    maxPixels=1e9
)

print("Temporal change statistics:")
print(change_stats.getInfo())

Temporal change statistics:
{'Delta_NDBI_max': 1.4798335472831852, 'Delta_NDBI_mean': -0.021196680002807677, 'Delta_NDBI_min': -0.8225616665904968, 'Delta_NDVI_max': 0.9100984376608621, 'Delta_NDVI_mean': -0.0029595245615970747, 'Delta_NDVI_min': -1.3466810923534513, 'Delta_NDWI_max': 1.3765394165459501, 'Delta_NDWI_mean': 0.010081431245611666, 'Delta_NDWI_min': -1.0429711352201503}
